# ETL Silver ➜ Gold FIFA 21 Players

Este notebook realiza a transformação dos dados da camada **Silver**
para a camada **Gold**, aplicando **modelagem dimensional (Star Schema)**.

### Tabelas Gold geradas:
- dim_ply (Jogador)
- dim_tm (Time)
- dim_pos (Posição)
- fat_ply_sts (Fato de atributos e ratings)


## Imports e Configurações

In [35]:
import os
import pandas as pd
import numpy as np

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

## Carrega variáveis de ambiente (.env)

In [36]:
load_dotenv("../.env")

DB_CONFIG = {
    'host': os.getenv('POSTGRES_HOST'),
    'port': int(os.getenv('POSTGRES_PORT')),
    'database': os.getenv('POSTGRES_DB'),
    'user': os.getenv('POSTGRES_USER'),
    'password': os.getenv('POSTGRES_PASSWORD')
}

DATABASE_URL = (
    f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

engine = create_engine(DATABASE_URL)

## Preparação da Camada Gold (DDL)

Antes de qualquer transformação ou carga, executamos o **DDL da camada Gold**.

Esse DDL é responsável por:
- Criar o schema `gold`
- Criar tabelas dimensão e fato
- Definir PKs, FKs, índices e comentários

Essa etapa garante:
- Integridade referencial
- Performance
- Padronização do modelo

In [37]:
DDL_PATH = "../Data Layer/gold/ddl.sql"

with open(DDL_PATH, "r") as file:
    ddl_sql = file.read()

with engine.begin() as conn:
    conn.execute(text(ddl_sql))

print("DDL da camada Gold executado com sucesso.")


DDL da camada Gold executado com sucesso.


## Leitura da Silver

Nesta etapa, os dados são lidos da camada Silver.

A Silver já contém dados:
- Limpos
- Tipados
- Padronizados

In [38]:
QUERY_SILVER = """
SELECT *
FROM silver.fifa21_players
"""

df_silver = pd.read_sql(QUERY_SILVER, engine)

df_silver.head()

,player_id,long_name,name,nationality,positions,age,overall_rating,potential_rating,team,contract_start_year,...,attack_work_rate,defense_work_rate,international_reputation,pace,shooting,passing,dribbling_stat,defending_stat,physical,hits
0,158023,Lionel Messi,L. Messi,Argentina,RW ST CF,33,93,93,FC Barcelona,2004.0,...,Medium,Low,5,85,92,91,95,38,65,372
1,20801,C. Ronaldo dos Santos Aveiro,Cristiano Ronaldo,Portugal,ST LW,35,92,92,Juventus,2018.0,...,High,Low,5,89,93,81,89,35,77,344
2,200389,Jan Oblak,J. Oblak,Slovenia,GK,27,91,93,Atlético Madrid,2014.0,...,Medium,Medium,3,87,92,78,90,52,90,86
3,192985,Kevin De Bruyne,K. De Bruyne,Belgium,CAM CM,29,91,91,Manchester City,2015.0,...,High,High,4,76,86,93,88,64,78,163
4,190871,Neymar da Silva Santos Jr.,Neymar Jr,Brazil,LW CAM,28,91,91,Paris Saint-Germain,2017.0,...,High,Medium,5,91,85,86,94,36,59,273


## Dimensões

As dimensões representam entidades descritivas do negócio.
Neste projeto:
- Jogador
- Time / Contrato
- Posição

Cada dimensão recebe uma **surrogate key**
utilizada posteriormente na tabela fato.

### Dimensão Jogador (dim_ply)

Contém informações demográficas e técnicas do jogador.

Cada jogador aparece **uma única vez** na dimensão.

In [39]:
dim_ply = (
    df_silver[[
        "player_id",
        "long_name",
        "name",
        "age",
        "height_cm",
        "weight_kg",
        "preferred_foot",
        "weak_foot",
        "skill_moves",
        "international_reputation",
        "nationality"
    ]]
    .drop_duplicates()
    .rename(columns={
        "player_id": "cod_ply",
        "long_name": "txt_nam_lng",
        "name": "nom_nam_sht",
        "age": "num_age",
        "height_cm": "num_hgt",
        "weight_kg": "num_wgt",
        "preferred_foot": "nom_fot",
        "weak_foot": "num_wkf",
        "skill_moves": "num_skm",
        "international_reputation": "num_rep",
        "nationality": "nom_nat"
    })
    .reset_index(drop=True)
)

dim_ply.insert(0, "srk_ply", range(1, len(dim_ply) + 1))


### Dimensão Time (dim_tm)

Representa o vínculo contratual do jogador com um time.

Nesta modelagem cada combinação de contrato é única.

In [40]:
dim_tm = (
    df_silver[[
        "team",
    ]]
    .drop_duplicates()
    .rename(columns={
        "team": "nom_tim"
    })
    .reset_index(drop=True)
)

dim_tm.insert(0, "srk_tim", range(1, len(dim_tm) + 1))


### Dimensão Posição (dim_pos)

Contém as posições jogáveis do atleta e sua posição principal.

In [41]:
dim_pos = (
    df_silver[[
        "positions",
        "best_position"
    ]]
    .drop_duplicates()
    .rename(columns={
        "positions": "nom_pos",
        "best_position": "nom_pos_bst"
    })
    .reset_index(drop=True)
)

dim_pos.insert(0, "srk_pos", range(1, len(dim_pos) + 1))


## Tabela Fato

A tabela fato concentra:
- Métricas
- Ratings
- Atributos quantitativos

Ela referencia as dimensões por meio de **foreign keys (surrogate keys)**.

### Mapeamento de Chaves (Lookups)

Aqui realizamos os joins entre a Silver e as dimensões
para substituir chaves naturais por surrogate keys.

In [42]:
df_fact = df_silver.copy()

df_fact = df_fact.merge(
    dim_ply[["srk_ply", "cod_ply"]],
    left_on="player_id",
    right_on="cod_ply",
    how="left"
)

df_fact = df_fact.merge(
    dim_tm[["srk_tim", "nom_tim"]],
    left_on="team",
    right_on="nom_tim",
    how="left"
)

df_fact = df_fact.merge(
    dim_pos[["srk_pos", "nom_pos", "nom_pos_bst"]],
    left_on=["positions", "best_position"],
    right_on=["nom_pos", "nom_pos_bst"],
    how="left"
)


### Construção da fat_ply_sts

Seleciona apenas as colunas relevantes para análise
e renomeia as chaves para o padrão SRK (Surrogate Key).

In [ ]:
fat_ply_sts = (
    df_fact[[
        "srk_ply",
        "srk_tim",
        "srk_pos",

        "overall_rating",
        "potential_rating",
        "best_overall_rating",
        "growth",
        "total_stats",
        "base_stats",
        "hits",

        "value_eur",
        "wage_eur",
        "release_clause_eur",

        "contract_start_year",
        "contract_end_year",
        "joined_date",

        "pace",
        "shooting",
        "passing",
        "dribbling_stat",
        "defending_stat",
        "physical",

        "attacking_total",
        "crossing",
        "finishing",
        "heading_accuracy",
        "short_passing",
        "volleys",

        "skill_total",
        "dribbling",
        "curve",
        "fk_accuracy",
        "long_passing",
        "ball_control",

        "movement_total",
        "acceleration",
        "sprint_speed",
        "agility",
        "reactions",
        "balance",

        "power_total",
        "shot_power",
        "jumping",
        "stamina",
        "strength",
        "long_shots",

        "mentality_total",
        "aggression",
        "interceptions",
        "positioning",
        "vision",
        "penalties",
        "composure",

        "defending_total",
        "marking",
        "standing_tackle",
        "sliding_tackle",

        "goalkeeping_total",
        "gk_diving",
        "gk_handling",
        "gk_kicking",
        "gk_positioning",
        "gk_reflexes"
    ]]
    .rename(columns={
        "overall_rating": "num_ovr",
        "potential_rating": "num_pot",
        "best_overall_rating": "num_bst_ovr",
        "growth": "num_gro",
        "total_stats": "num_tot",
        "base_stats": "num_bas",
        "hits": "num_hit",

        "value_eur": "vlr_mkt",
        "wage_eur": "vlr_wag",
        "release_clause_eur": "vlr_rel",

        "contract_start_year": "num_str_yr",
        "contract_end_year": "num_end_yr",
        "joined_date": "dat_joi",

        "pace": "num_pac",
        "shooting": "num_sho",
        "passing": "num_pas",
        "dribbling_stat": "num_dri",
        "defending_stat": "num_def",
        "physical": "num_phy",

        "attacking_total": "num_att_tot",
        "crossing": "num_att_crs",
        "finishing": "num_att_fin",
        "heading_accuracy": "num_att_hed",
        "short_passing": "num_att_pas",
        "volleys": "num_att_vol",

        "skill_total": "num_skl_tot",
        "dribbling": "num_skl_dri",
        "curve": "num_skl_cur",
        "fk_accuracy": "num_skl_fka",
        "long_passing": "num_skl_lps",
        "ball_control": "num_skl_ctl",

        "movement_total": "num_mov_tot",
        "acceleration": "num_mov_acc",
        "sprint_speed": "num_mov_spr",
        "agility": "num_mov_agi",
        "reactions": "num_mov_rea",
        "balance": "num_mov_bal",

        "power_total": "num_pow_tot",
        "shot_power": "num_pow_sht",
        "jumping": "num_pow_jmp",
        "stamina": "num_pow_sta",
        "strength": "num_pow_str",
        "long_shots": "num_pow_lsh",

        "mentality_total": "num_men_tot",
        "aggression": "num_men_agg",
        "interceptions": "num_men_int",
        "positioning": "num_men_pos",
        "vision": "num_men_vis",
        "penalties": "num_men_pen",
        "composure": "num_men_cmp",

        "defending_total": "num_def_tot",
        "marking": "num_def_mrk",
        "standing_tackle": "num_def_sta",
        "sliding_tackle": "num_def_slt",

        "goalkeeping_total": "num_gkp_tot",
        "gk_diving": "num_gkp_div",
        "gk_handling": "num_gkp_han",
        "gk_kicking": "num_gkp_kic",
        "gk_positioning": "num_gkp_pos",
        "gk_reflexes": "num_gkp_ref"
    })
)

fat_ply_sts.insert(0, "srk_ply_sts", range(1, len(fat_ply_sts) + 1))


### Validações Básicas

Garante que todos os registros da fato
possuem relacionamento válido com as dimensões.

In [44]:
assert fat_ply_sts["srk_ply"].isnull().sum() == 0
assert fat_ply_sts["srk_tim"].isnull().sum() == 0
assert fat_ply_sts["srk_pos"].isnull().sum() == 0

## Escrita das tabelas Gold no banco

As tabelas são carregadas no schema `gold`.

Utilizamos `append` pois o schema já foi criado via DDL.

In [45]:
dim_ply.to_sql(
    "dim_ply",
    engine,
    schema="dw",
    if_exists="append",
    index=False
)

dim_tm.to_sql(
    "dim_tim",
    engine,
    schema="dw",
    if_exists="append",
    index=False
)

dim_pos.to_sql(
    "dim_pos",
    engine,
    schema="dw",
    if_exists="append",
    index=False
)

fat_ply_sts.to_sql(
    "fat_ply_sts",
    engine,
    schema="dw",
    if_exists="append",
    index=False
)


ProgrammingError: (psycopg2.errors.UndefinedColumn) column "attacking_total" of relation "fat_ply_sts" does not exist
LINE 1: ...pac, num_sho, num_pas, num_dri, num_def, num_phy, attacking_...
                                                             ^

[SQL: INSERT INTO dw.fat_ply_sts (srk_ply_sts, srk_ply, srk_tim, srk_pos, num_ovr, num_pot, num_bst_ovr, num_gro, num_tot, num_bas, num_hit, vlr_mkt, vlr_wag, vlr_rel, num_str_yr, num_end_yr, dat_joi, num_pac, num_sho, num_pas, num_dri, num_def, num_phy, a ... 666074 characters truncated ... ng__509)s, %(gk_handling__509)s, %(gk_kicking__509)s, %(gk_positioning__509)s, %(gk_reflexes__509)s)]
[parameters: {'num_ovr__0': 93, 'dribbling__0': 96, 'reactions__0': 94, 'balance__0': 95, 'num_end_yr__0': 2021.0, 'acceleration__0': 91, 'penalties__0': 75, 'vlr_wag__0': 560000.0, 'num_pot__0': 93, 'vlr_rel__0': 138400000.0, 'agility__0': 91, 'strength__0': 69, 'finishing__0': 95, 'short_passing__0': 91, 'srk_tim__0': 1, 'curve__0': 93, 'num_bs

## Exportação CSV da Gold Fato

Exportação opcional da tabela fato para uso externo,
validação local.

In [ ]:
OUTPUT_DIR = "../Data Layer/gold"
OUTPUT_FILE_CSV = "fifa21_gold.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE_CSV)

fat_ply_sts.to_csv(output_path, index=False)

print(f"CSV Gold gerado em: {output_path}")

CSV Gold gerado em: ../Data Layer/gold/fifa21_gold.csv


## Validação pós-carga

Verificação simples para garantir que os dados
foram persistidos corretamente na camada Gold.

In [ ]:
pd.read_sql(
    "SELECT COUNT(*) FROM dw.fat_ply_sts",
    engine
)

,count
0,18978
